In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import kagglehub

## 1. Выбор начальных условий

### Выбор датасета

Был выбран датасет **Facial Emotion Recognition**, содержащий изображения 7 эмоций:

| Эмоция |
|---|
|  Злость |
| Отвращение |
|  Страх |
|  Счастье |
|  Нейтрально |
|  Грусть |
|  Удивление |

**Практическая значимость:** Распознавание эмоций может быть полезно для развития технологий симуляции эмоций в компьютерной графике и в конструировании человекоподобных роботов. Также в будущем нейросети могут использовать распознавание эмоций для лучшего понимания контекста в диалоге с пользователем.

###  Метрики качества

Для оценки качества моделей выбраны следующие метрики:
1. **Accuracy (Точность)** — базовая метрика, показывающая общую долю правильных ответов: \
   $Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$
2. **F1-Score (Макро-усреднение)** — гармоническое среднее между Precision и Recall: \
   $F1 = \frac{2 \cdot Precision \cdot Recall}{Precision + Recall}$ \
*Обоснование:* Датасеты с эмоциями часто бывают несбалансированными (например, "счастье" встречается чаще, чем "отвращение"). Метрика Accuracy может давать искаженное представление об успешности модели на классах меньшинства. F1-score с параметром `macro` позволит объективно оценить, насколько хорошо модель распознает каждую конкретную эмоцию независимо от размера класса.

---
## 2. Создание бейзлайна и оценка качества

Загрузка датасета

In [ ]:
DATA_DIR = kagglehub.dataset_download("fahadullaha/facial-emotion-recognition-dataset")
DATA_DIR = os.path.join(DATA_DIR, "processed_data")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Path to dataset files:", DATA_DIR)
print(f"Используемое устройство: {device}")

Using Colab cache for faster access to the 'facial-emotion-recognition-dataset' dataset.
Path to dataset files: /kaggle/input/facial-emotion-recognition-dataset/processed_data
Используемое устройство: cuda


Для бейзлайна используем стандартные преобразования изображений, без сложных аугментаций, только приведение к размеру 224x224, так как этого требуют многие модели из `torchvision`, и нормализация. \
Модели бейзлайна: Сверточная сеть (ResNet18) и Трансформер (Vision Transformer - ViT_B_16).

In [ ]:
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=base_transform)

num_classes = len(full_dataset.classes)
print(f"Найдено классов: {num_classes} ({full_dataset.classes})")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

Найдено классов: 7 (['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'])


In [ ]:
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")

    return acc, f1

criterion = nn.CrossEntropyLoss()

In [ ]:
print("--- Запуск бейзлайна: ResNet18 ---")
resnet_base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_ftrs = resnet_base.fc.in_features
resnet_base.fc = nn.Linear(num_ftrs, num_classes)
resnet_base = resnet_base.to(device)

optimizer_resnet = optim.Adam(resnet_base.parameters(), lr=0.001)

acc_res_base, f1_res_base = train_and_evaluate(resnet_base, train_loader, val_loader, criterion, optimizer_resnet, epochs=5)

--- Запуск бейзлайна: ResNet18 ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 160MB/s]


Epoch 1/5 | Train Loss: 1.1340 | Val Acc: 0.6208 | Val F1: 0.5916
Epoch 2/5 | Train Loss: 0.9026 | Val Acc: 0.6806 | Val F1: 0.6551
Epoch 3/5 | Train Loss: 0.7897 | Val Acc: 0.6864 | Val F1: 0.6544
Epoch 4/5 | Train Loss: 0.6782 | Val Acc: 0.7079 | Val F1: 0.6895
Epoch 5/5 | Train Loss: 0.5392 | Val Acc: 0.7094 | Val F1: 0.6824


In [ ]:
print("\n--- Запуск бейзлайна: Vision Transformer (ViT_B_16) ---")
vit_base = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in vit_base.parameters():
    param.requires_grad = False

num_ftrs_vit = vit_base.heads.head.in_features
vit_base.heads.head = nn.Linear(num_ftrs_vit, num_classes)
vit_base = vit_base.to(device)

optimizer_vit = optim.Adam(vit_base.parameters(), lr=1e-5)

acc_vit_base, f1_vit_base = train_and_evaluate(vit_base, train_loader, val_loader, criterion, optimizer_vit, epochs=5)


--- Запуск бейзлайна: Vision Transformer (ViT_B_16) ---
Epoch 1/5 | Train Loss: 1.2800 | Val Acc: 0.5597 | Val F1: 0.5325
Epoch 2/5 | Train Loss: 1.1435 | Val Acc: 0.5757 | Val F1: 0.5381
Epoch 3/5 | Train Loss: 1.1037 | Val Acc: 0.5842 | Val F1: 0.5522
Epoch 4/5 | Train Loss: 1.0812 | Val Acc: 0.5807 | Val F1: 0.5599
Epoch 5/5 | Train Loss: 1.0684 | Val Acc: 0.5880 | Val F1: 0.5623


Как видим ResNet18 справился лучше ViT_B_16. Всё из-за того, что ViT требует гораздо больше данных для обобщения, чем сверточные сети. Трансформеры делят картинку на патчи и ищут глобальные взаимосвязи между ними с помощью механизма внимания. У них нет встроенного понимания локальной структуры изображения.

## 3. Улучшение бейзлайна

**Формулировка гипотез** \
Гипотеза 1: Применение аугментации данных (случайные повороты, отзеркаливания и изменения яркости) поможет модели лучше обобщать данные и меньше переобучаться на тренировочной выборке. \
Гипотеза 2: Использование планировщика скорости обучения (Learning Rate Scheduler) позволит алгоритму точнее "сойтись" в локальный минимум функции потерь, уменьшая шаг на поздних эпохах.

**Проверка гипотез и формирование улучшенного бейзлайна** \
Мы добавим `RandomHorizontalFlip` и `RandomRotation` в `transforms` для тренировочного датасета. Также добавим `StepLR` в процесс обучения.

In [ ]:
augmented_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_train_wrapper = torch.utils.data.Subset(datasets.ImageFolder(root=DATA_DIR, transform=augmented_transform), train_dataset.indices)
dataset_val_wrapper = torch.utils.data.Subset(datasets.ImageFolder(root=DATA_DIR, transform=base_transform), val_dataset.indices)

train_loader_aug = DataLoader(dataset_train_wrapper, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader_aug = DataLoader(dataset_val_wrapper, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


print("--- Запуск улучшенного бейзлайна: ResNet18 + Augmentations + LR Scheduler ---")

resnet_improved = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet_improved.fc = nn.Linear(resnet_improved.fc.in_features, num_classes)
resnet_improved = resnet_improved.to(device)

optimizer_imp = optim.Adam(resnet_improved.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer_imp, step_size=3, gamma=0.1)

def train_and_evaluate_with_scheduler(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')
        print(f"Epoch {epoch+1}/{epochs} | LR: {scheduler.get_last_lr()[0]:.6f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")
    return acc, f1

acc_res_imp, f1_res_imp = train_and_evaluate_with_scheduler(
    resnet_improved,
    train_loader_aug,
    val_loader_aug,
    criterion,
    optimizer_imp,
    scheduler,
    epochs=5
)

--- Запуск улучшенного бейзлайна: ResNet18 + Augmentations + LR Scheduler ---
Epoch 1/5 | LR: 0.001000 | Val Acc: 0.6173 | Val F1: 0.5847
Epoch 2/5 | LR: 0.001000 | Val Acc: 0.6694 | Val F1: 0.6451
Epoch 3/5 | LR: 0.000100 | Val Acc: 0.6567 | Val F1: 0.6404
Epoch 4/5 | LR: 0.000100 | Val Acc: 0.7338 | Val F1: 0.7121
Epoch 5/5 | LR: 0.000100 | Val Acc: 0.7459 | Val F1: 0.7248


In [ ]:
print("\n--- Запуск улучшенного бейзлайна: ViT_B_16 + Augmentations + LR Scheduler ---")

vit_improved = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in vit_improved.parameters():
    param.requires_grad = False

num_ftrs_vit_imp = vit_improved.heads.head.in_features
vit_improved.heads.head = nn.Linear(num_ftrs_vit_imp, num_classes)
vit_improved = vit_improved.to(device)

optimizer_vit_imp = optim.Adam(vit_improved.heads.head.parameters(), lr=0.001)

scheduler_vit = optim.lr_scheduler.StepLR(optimizer_vit_imp, step_size=3, gamma=0.1)

acc_vit_imp, f1_vit_imp = train_and_evaluate_with_scheduler(
    vit_improved,
    train_loader_aug,
    val_loader_aug,
    criterion,
    optimizer_vit_imp,
    scheduler_vit,
    epochs=5
)


--- Запуск улучшенного бейзлайна: ViT_B_16 + Augmentations + LR Scheduler ---
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:06<00:00, 50.2MB/s]


Epoch 1/5 | LR: 0.001000 | Val Acc: 0.5374 | Val F1: 0.4908
Epoch 2/5 | LR: 0.001000 | Val Acc: 0.5698 | Val F1: 0.5394
Epoch 3/5 | LR: 0.000100 | Val Acc: 0.5675 | Val F1: 0.5360
Epoch 4/5 | LR: 0.000100 | Val Acc: 0.5869 | Val F1: 0.5589
Epoch 5/5 | LR: 0.000100 | Val Acc: 0.5810 | Val F1: 0.5490


**Сравнение результатов и Выводы**

Сравнивая результаты ResNet18 Base и ResNet18 Improved, можно сделать вывод, что гипотеза подтвердилась.
| Метрика | ResNet18 | ResNet18 Improved |
|---------|----------|-------------------|
| Accuray | 0.7094   | 0.7459 |
| F1-score | 0.6824 | 0.7248 |

Аугментации помогают модели не переобучиться. Снижение Learning Rate на 3-й эпохе позволило алгоритму точнее скорректировать веса, что отразилось в резком росте метрик на 4-й эпохе.

При сравнении ViT_B_16 и ViT_B_16 Improved можно сделать вывод, что гепотеза не подтвердилась. Наблюдается падение метрик.
| Метрика | ViT_B_16 | ViT_B_16 Improved |
|---------|----------|-------------------|
| Accuray | 0.5880   | 0.5810 |
| F1-score | 0.5623 | 0.5490 |



## 4. Имплементация алгоритма машинного обучения

**4.a) Самостоятельная имплементация модели** \
Мы создадим собственную сверточную нейронную сеть (CustomCNN) с нуля, используя `torch.nn.Module`. Архитектура будет состоять из трех сверточных блоков (Conv2d + ReLU + MaxPool2d) и полносвязного слоя.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc = nn.Linear(64 * 28 * 28, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
print("--- Запуск кастомной CNN на базовых данных ---")
custom_model_base = CustomCNN(num_classes).to(device)
optimizer_custom = optim.Adam(custom_model_base.parameters(), lr=0.001)

acc_custom_base, f1_custom_base = train_and_evaluate(custom_model_base, train_loader, val_loader, criterion, optimizer_custom, epochs=5)

--- Запуск кастомной CNN на базовых данных ---
Epoch 1/5 | Train Loss: 1.3596 | Val Acc: 0.5464 | Val F1: 0.5004
Epoch 2/5 | Train Loss: 1.0793 | Val Acc: 0.5699 | Val F1: 0.5365
Epoch 3/5 | Train Loss: 0.9460 | Val Acc: 0.5629 | Val F1: 0.5271
Epoch 4/5 | Train Loss: 0.8243 | Val Acc: 0.5801 | Val F1: 0.5478
Epoch 5/5 | Train Loss: 0.7020 | Val Acc: 0.5737 | Val F1: 0.5428


In [ ]:
print("\n--- Запуск кастомной CNN на улучшенных данных (Аугментации + Scheduler) ---")
custom_model_imp = CustomCNN(num_classes).to(device)
optimizer_custom_imp = optim.Adam(custom_model_imp.parameters(), lr=0.001)
scheduler_custom = optim.lr_scheduler.StepLR(optimizer_custom_imp, step_size=3, gamma=0.1)

acc_custom_imp, f1_custom_imp = train_and_evaluate_with_scheduler(
    custom_model_imp, train_loader_aug, val_loader_aug, criterion, optimizer_custom_imp, scheduler_custom, epochs=5
)


--- Запуск кастомной CNN на улучшенных данных (Аугментации + Scheduler) ---
Epoch 1/5 | LR: 0.001000 | Val Acc: 0.4834 | Val F1: 0.4347
Epoch 2/5 | LR: 0.001000 | Val Acc: 0.5536 | Val F1: 0.5279
Epoch 3/5 | LR: 0.000100 | Val Acc: 0.5559 | Val F1: 0.5248
Epoch 4/5 | LR: 0.000100 | Val Acc: 0.5846 | Val F1: 0.5568
Epoch 5/5 | LR: 0.000100 | Val Acc: 0.5890 | Val F1: 0.5565


**4.a) Самостоятельная имплементация трансформерной модели (Custom ViT)**

Для имплементации Vision Transformer создается архитектура "Mini-ViT". Она включает:
1. **Patch Embedding:** Изображение 224x224 разбивается на патчи (например, 32x32 пикселя) с помощью сверточного слоя. К каждому патчу добавляется позиционное кодирование (обучаемые параметры), чтобы модель понимала, где находится патч, а также специальный токен классификации (CLS-токен).
2. **Transformer Encoder:** Набор слоев с механизмом Multi-Head Self-Attention.
3. **MLP Head:** Полносвязный слой, который берет на входе только CLS-токен, прошедший через трансформер, и выдает предсказания классов.

In [ ]:
class CustomViT(nn.Module):
    def __init__(self, num_classes, img_size=224, patch_size=32, dim=128, depth=3, heads=4, mlp_dim=256, dropout=0.1):
        super().__init__()
        assert img_size % patch_size == 0,
        num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)

        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.norm = nn.LayerNorm(dim)
        self.mlp_head = nn.Linear(dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x).flatten(2).transpose(1, 2)

        cls = self.cls_token.expand(B, -1, -1)

        x = torch.cat([cls, x], dim=1)

        x = self.dropout(x + self.pos_embedding)

        x = self.transformer(x)

        cls_out = self.norm(x[:, 0])
        return self.mlp_head(cls_out)

In [ ]:
print("--- Запуск кастомного ViT на базовых данных ---")
custom_vit_base = CustomViT(num_classes).to(device)

optimizer_custom_vit = optim.Adam(custom_vit_base.parameters(), lr=0.0003)

acc_custom_vit_base, f1_custom_vit_base = train_and_evaluate(
    custom_vit_base, train_loader, val_loader, criterion, optimizer_custom_vit, epochs=5
)

--- Запуск кастомного ViT на базовых данных ---


/tmp/ipykernel_2211/2590306422.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)


Epoch 1/5 | Train Loss: 1.6963 | Val Acc: 0.3483 | Val F1: 0.2642
Epoch 2/5 | Train Loss: 1.5981 | Val Acc: 0.3628 | Val F1: 0.2935
Epoch 3/5 | Train Loss: 1.5330 | Val Acc: 0.4126 | Val F1: 0.3589
Epoch 4/5 | Train Loss: 1.4779 | Val Acc: 0.4167 | Val F1: 0.3467
Epoch 5/5 | Train Loss: 1.4479 | Val Acc: 0.4383 | Val F1: 0.3854


In [ ]:
print("\n--- Запуск кастомного ViT на улучшенных данных (Аугментации + Scheduler) ---")
custom_vit_imp = CustomViT(num_classes).to(device)

optimizer_custom_vit_imp = optim.Adam(custom_vit_imp.parameters(), lr=0.0003)
scheduler_custom_vit = optim.lr_scheduler.StepLR(optimizer_custom_vit_imp, step_size=3, gamma=0.1)

acc_custom_vit_imp, f1_custom_vit_imp = train_and_evaluate_with_scheduler(
    custom_vit_imp, train_loader_aug, val_loader_aug, criterion, optimizer_custom_vit_imp, scheduler_custom_vit, epochs=5
)


--- Запуск кастомного ViT на улучшенных данных (Аугментации + Scheduler) ---


/tmp/ipykernel_2211/2590306422.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)


Epoch 1/5 | LR: 0.000300 | Val Acc: 0.2406 | Val F1: 0.1824
Epoch 2/5 | LR: 0.000300 | Val Acc: 0.2845 | Val F1: 0.2084
Epoch 3/5 | LR: 0.000030 | Val Acc: 0.2316 | Val F1: 0.1903
Epoch 4/5 | LR: 0.000030 | Val Acc: 0.3042 | Val F1: 0.2905
Epoch 5/5 | LR: 0.000030 | Val Acc: 0.3428 | Val F1: 0.2864


**Сравнение результатов и финальные выводы**

1. **Кастомная сеть vs Предобученный Бейзлайн:** Предобученные модели (ResNet, ViT), показывают значительно лучшие результаты по сравнению с написанной с нуля неглубокой сетью. Это связано с тем, что модели из `torchvision` уже обучены извлекать общие паттерны на миллионах изображений.

    | Метрика  | ResNet | CustomCNN |
    |----------|--------|-----------|
    | Accuracy | 0.7094 | 0.5737    |
    | F1-score | 0.6824 | 0.5428    |

    | Метрика  | ViT_B_16 | CustomViT |
    |----------|----------|-----------|
    | Accuracy | 0.5880   | 0.4383    |
    | F1-score | 0.5623   | 0.3854    |

2. **Кастомная сеть (база) vs Кастомная сеть (улучшенная):** В общем результаты такие же как и у предобученных моделей. CustomCNN показал улучшение результатов после введения аугментаций и scheduler. CustomViT показал падение метрик. Связано это со архитектурой трансформенных сетей. Аугментации и scheduler борятся с переобучением, но трансформенным сетям наоборот нужно больше данных чтобы просто уловить закономерности. Поэтому аугментации только портят картину, а действительно может помочь только увеличение датасета и количества эпох, разморозка бóльших слоев, уменьшение размера патча. Но на это нужно больше вычислительных ресурсов, которых к сожалению у меня нет.

    | Метрика  | CustomCNN | CustomCNN_improved |
    |----------|-----------|--------------------|
    | Accuracy | 0.5737    | 0.5890             |
    | F1-score | 0.5428    | 0.5565             |

    | Метрика  | CustomViT | CustomViT_improved |
    |----------|-----------|--------------------|
    | Accuracy | 0.4383    | 0.3428            |
    | F1-score | 0.3854    | 0.2864            |

3. **Общий вывод:** Использование предобученных моделей является стандартом де-факто для задач компьютерного зрения. Однако, грамотная работа с данными аугментации и гиперпараметрами важна вне зависимости от архитектуры.

## Общие выводы по лабораторной работе

Задача автоматической классификации эмоций характеризуется высокой степенью субъективности. Разметка визуальных данных в этой предметной области сильно подвержена влиянию человеческого фактора, так как мимические проявления часто неоднозначны. Статистика показывает, что на популярных эталонных наборах данных, таких как FER-2013, базовая точность распознавания эмоций человеком составляет лишь около 65%. В связи с этим достигнутые моделями метрики являются адекватным результатом, отражающим реальную сложность предметной области.

В ходе выполнения работы было проведено сравнение архитектур сверточных нейронных сетей и моделей на базе механизма внимания. Сверточная архитектура продемонстрировала более высокую точность распознавания и стабильность при обучении. Применение методов улучшения бейзлайна, таких как аугментация данных и использование планировщика скорости обучения, привело к прогнозируемому росту метрик качества для модели ResNet18.

Гипотеза об улучшении качества трансформерной модели за счет аугментации данных не подтвердилась. Введение геометрических и цветовых искажений в обучающую выборку привело к снижению метрик Accuracy и F1-score для архитектуры ViT.

Данный результат обусловлен отсутствием у трансформеров встроенного индуктивного смещения для поиска локальных пространственных признаков. Для формирования связей между патчами трансформерам требуются масштабные наборы данных. При работе с ограниченным датасетом и использовании замороженных базовых весов дополнительная аугментация выступает в роли избыточного шума, который препятствует сходимости алгоритма.

Использование предобученных сверточных сетей является наиболее эффективным подходом для решения задач компьютерного зрения на небольших наборах данных. Обучение сложных трансформерных архитектур в условиях ограниченных вычислительных мощностей и малого объема выборки нецелесообразно.